In [2]:
# set up autoreload on all files
%load_ext autoreload
%autoreload 2
# step up folder hiearchy to adjust module location for import
import sys
sys.path.insert(0, '..')
%cd ..\\..

D:\jkerssemakers\Dropbox\CD_recent


In [3]:
from os import path
from glob import glob
import numpy as np
import matplotlib.pyplot as plt
from itertools import chain
from pygbox.viz import viz, plots
from pygbox.fio.name import make_fpath
from pygbox.pygbox import StackContainer, Quantifier, Stack, Segmentor

%matplotlib qt

## Load example data

In [4]:
files = glob('U:\gbox\develop\sidetraps_examples\*.json')
tiffpaths = glob('U:\gbox\develop\sidetraps_examples\*.tif')
maskpaths = glob('U:\gbox\develop\sidetraps_examples\*Segmentor.np*')

### Demonstration of loading processed data

In [5]:
quants = []
stacks = []
masks = []
pix2um = (.118, .118, 0)
dt = 10/60 # min

for f, tif, msk in zip(files, tiffpaths, maskpaths):
    q = Quantifier()
    q.load(f)
    quants.append(q)
    
    s = Stack(fpath = tif, pix2um = pix2um)
    s.load()
    stacks.append(s)

    seg = Segmentor()
    seg.load(fpath = msk)
    masks.append(seg.mask)

In [44]:
viz.imshow([], stacks[0].im*masks[0])

<Axes: >

In [90]:
stacks[0].im.shape

(229, 226, 1, 78)

In [45]:
#get rdius of gyration (or any other operation) along time axis
from pygbox import ops
for i in range(stacks[0].im.shape[-1]):
    abc = ops.radius_of_gyration(stacks[0].im[..., i], masks[0][..., i], pix2um, False)
    #print(abc)

# fluorescence histograms
Below, values of non-masked pixels for each frame are counted in a histogram. Coloring is per time point. To improve statistics, multiple frames can be collected (thus, a histogram refers to the former n... frames)

In [105]:
#set up fluorescence loop
nbins=25
minbin=0
maxbin=10000
binax = np.linspace(minbin, maxbin, nbins)
midax=binax[0:-1]+np.diff(binax)
ff=stacks[0].im.shape[-1]

#set up imaging sequence:
collect_frames=10
plt.close("all")


In [106]:
for sti in range(len(stacks)):
    collected_pixels=[]
    frame_index = 0
    Fluorescence_all = []
    pixels_histmap = np.zeros((ff, nbins-1))
    fig, axs = plt.subplots(1, 2)
    cm = plt.get_cmap('inferno')
    NUM_COLORS0 = int(ff)
    NUM_COLORS1 = int(ff)
    axs[0].set_prop_cycle(color=[cm(1.*ci/NUM_COLORS0) for ci in range(NUM_COLORS0)])
    axs[1].set_prop_cycle(color=[cm(1.*ci/NUM_COLORS1) for ci in range(NUM_COLORS1)])
    for i in range(stacks[0].im.shape[-1]):
        print(sti, i, FL)        
        #work_im=stacks[sti].im[..., 0,  i]*masks[sti][..., 0,  i]
        work_im=stacks[sti].im[..., 0,  i] 
        #work only on non-masked area
        FL=np.sum(np.array(work_im[np.nonzero(work_im > 0)]))
        Fluorescence_all.append(FL)           
        # histograms per frame from inner circle
        collected_pixels.append(np.array(work_im[np.nonzero(work_im > 0)]))    
        if (np.mod(frame_index, collect_frames) == 0):
            pixels_hist_thisframe, edges = np.histogram(collected_pixels, binax)
            collected_pixels=[]           
            axs[1].loglog(midax,pixels_hist_thisframe, '-')
            axs[1].set_title("histogram")
            axs[1].set_xlabel("intensity (a.u.)")
            axs[1].set_ylabel("counts (a.u.)")
            axs[1].set_xlim(100, maxbin)
            axs[1].set_ylim(1, 1E4)
            fig.tight_layout()
                
    axs[0].set_title("fluorescence count")
    axs[0].set_xlabel("time (mins)")
    axs[0].set_ylabel("counts (a.u.)")   
    for fi in range(ff) :
        axs[0].plot(fi*dt, Fluorescence_all[fi], "o") 
plt.show()

0 0 18185780
0 1 20202609
0 2 20067874
0 3 19937500
0 4 19928039
0 5 19845024
0 6 19777533
0 7 19700438
0 8 19740388
0 9 19726112
0 10 19613223
0 11 19599449
0 12 20038431
0 13 20056548
0 14 20172466
0 15 20073308
0 16 19961504
0 17 20067606
0 18 20041786
0 19 20016029
0 20 19967571
0 21 19804732
0 22 19762799
0 23 19860299
0 24 19760093
0 25 19754822
0 26 19819342
0 27 19784118
0 28 19796149
0 29 19685686
0 30 19691718
0 31 19525325
0 32 18018929
0 33 17673103
0 34 21161742
0 35 24496145
0 36 25962664
0 37 26509804
0 38 26369997
0 39 26517119
0 40 26637051
0 41 26448492
0 42 26619562
0 43 26598830
0 44 26590977
0 45 26740582
0 46 26592386
0 47 26622546
0 48 26504858
0 49 26383024
0 50 26484378
0 51 26414719
0 52 26450792
0 53 26457165
0 54 26492696
0 55 26249449
0 56 26564453
0 57 26591330
0 58 26369112
0 59 26257409
0 60 26215885
0 61 26348892
0 62 26218879
0 63 26070117
0 64 25981122
0 65 25931407
0 66 25863613
0 67 25975110
0 68 25993318
0 69 25814541
0 70 25846071
0 71 25703533
0 